# insurance-whittaker

**Whittaker-Henderson smoothing for insurance pricing tables.**

This notebook shows the core workflow in under two minutes:
generate a noisy age-frequency curve, smooth it with automatic lambda selection,
and read off the smoothed relativities with Bayesian credible intervals.

The problem this solves: every UK motor pricing actuary smooths experience tables.
The Whittaker-Henderson method has been actuarial best practice since 1923.
Most UK teams still do it in Excel or SAS. This library brings it to Python
with automatic lambda selection (REML) and proper uncertainty quantification.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-whittaker/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-whittaker polars numpy

## Synthetic experience data

We build a motor claim frequency table by age band: 17 to 79.
The true underlying curve is a declining exponential (young drivers are riskier).
We observe it with Poisson noise, which creates the jagged step pattern
you see in raw experience data before smoothing.

In [ ]:
import numpy as np
import polars as pl

rng = np.random.default_rng(2024)
ages = np.arange(17, 80)

# True underlying frequency: young drivers 3x more likely to claim
true_freq = 0.06 + 0.18 * np.exp(-(ages - 17) / 12)

# Exposures: sparse at extremes (few 17-year-olds, few 78-year-olds on book)
exposures = 50 + 800 * np.exp(-((ages - 38) ** 2) / 400)
exposures = exposures.astype(float)

# Observed claims: Poisson noise
observed_claims = rng.poisson(true_freq * exposures)
observed_freq   = observed_claims / exposures

df = pl.DataFrame({
    "age":       ages,
    "exposure":  exposures,
    "claims":    observed_claims,
    "obs_freq":  observed_freq,
})
print(f"{len(df)} age bands | total exposure: {exposures.sum():.0f} vehicle-years")
df.head(8)

## 1-D smoothing with automatic lambda

`WhittakerHenderson1D` applies penalised least squares smoothing.
The penalty order `q=2` penalises second differences — it biases toward
linear rather than constant fits, which is appropriate for age curves
that should be monotone over wide ranges.

We let the library choose `lambda` automatically via REML
(Restricted Maximum Likelihood). No manual tuning.

In [ ]:
from insurance_whittaker import WhittakerHenderson1D

wh = WhittakerHenderson1D(order=2, lambda_method="reml")
result = wh.fit(ages, observed_freq, weights=exposures)

print(f"Selected lambda: {result.lambda_:.1f}")
print(f"Effective degrees of freedom: {result.edf_:.1f}")

# View smoothed values and 95% credible intervals
smoothed_df = pl.DataFrame({
    "age":        ages,
    "obs_freq":   np.round(observed_freq, 4),
    "smoothed":   np.round(result.fitted, 4),
    "lower_95":   np.round(result.lower, 4),
    "upper_95":   np.round(result.upper, 4),
    "relativity": np.round(result.fitted / result.fitted.mean(), 3),
})
smoothed_df.head(10)

## Poisson smoother (for count data)

`WhittakerHendersonPoisson` fits the smooth curve directly to claim counts
rather than derived loss ratios. This is strictly more correct when exposures
vary — it weights the likelihood properly under a Poisson model.

Use this when your cells have low expected counts (thin data bands).

In [ ]:
from insurance_whittaker import WhittakerHendersonPoisson

wh_pois = WhittakerHendersonPoisson(order=2, lambda_method="reml")
result_pois = wh_pois.fit(ages, observed_claims, exposures=exposures)

print(f"Poisson smoother — lambda: {result_pois.lambda_:.1f}, EDF: {result_pois.edf_:.1f}")

# Compare WH Gaussian vs Poisson extension at young ages
comparison = pl.DataFrame({
    "age":          ages[:10],
    "true_freq":    np.round(true_freq[:10], 4),
    "wh_gaussian":  np.round(result.fitted[:10], 4),
    "wh_poisson":   np.round(result_pois.fitted[:10], 4),
})
print("\nYoung driver bands (where data is sparse):")
print(comparison)

## 2-D smoothing (age × vehicle group)

`WhittakerHenderson2D` smooths a cross-table simultaneously in both
dimensions. The practical use case: age × vehicle group, NCD × age,
or any factor interaction table you want to smooth without modelling
the interaction explicitly in a GLM.

In [ ]:
from insurance_whittaker import WhittakerHenderson2D

# 15 age bands × 8 vehicle groups
age_bands = np.arange(17, 32)  # 15 bands
veh_groups = np.arange(1, 9)   # 8 groups
rng2 = np.random.default_rng(99)

# True surface: young + high-group is most expensive
AA, VV = np.meshgrid(age_bands, veh_groups, indexing="ij")
true_surface = 0.05 + 0.15 * np.exp(-(AA - 17) / 5) + 0.01 * VV
exp_surface = rng2.poisson(200, (15, 8)).astype(float)
obs_surface = rng2.poisson(true_surface * exp_surface) / exp_surface

wh2 = WhittakerHenderson2D(order=(2, 1), lambda_method="reml")
result2 = wh2.fit(obs_surface, weights=exp_surface)

print(f"2-D smoother — lambda: ({result2.lambda_[0]:.1f}, {result2.lambda_[1]:.1f})")
print(f"Max abs error vs true: {np.abs(result2.fitted - true_surface).max():.4f}")
print(f"Smoothed surface shape: {result2.fitted.shape}  (age_bands × veh_groups)")

## What you should see

- The 1-D smoother recovers the declining age curve from noisy data.
  Credible intervals are wider at age 17 and 78 where exposure is thin.
- `WhittakerHendersonPoisson` and the Gaussian smoother agree closely
  in the dense centre bands; the Poisson version is more conservative
  at the sparse young-driver tail.
- The 2-D smoother handles the age × vehicle group surface in a single call,
  selecting different lambda values in each dimension.

## Next steps

The full library also includes:

- **`selection.py`** — compare REML, GCV, AIC, BIC lambda choices side by side
- **`plots.py`** — one-line smoothing diagnostic plots (requires matplotlib)
- **Benchmark notebook** — comparison against R's `WH` package on 100 real-world
  actuarial problems from Biessy (2026)

**GitHub:** https://github.com/burning-cost/insurance-whittaker  
**PyPI:** https://pypi.org/project/insurance-whittaker/